# What factors are linked to a student being at risk of dropping out?

**Outcome:** `Dropout_Risk` (Yes/No). `Burnout_Level` is treated as a secondary outcome (Q2) and as a possible factor for dropout.

**Questions**
1. Do students with higher family support have lower dropout risk?
2. How are sleep, exercise, social activity, study hours and screen time linked to burnout level?
3. Do students with part-time jobs study less, and are they more likely to be at dropout risk?
4. Are students with counseling access more or less at risk of dropping out?
5. Are students with more backlogs more likely to be at dropout risk?

**Note:** This is a synthetic (computer-generated) dataset, not real student records: [Student Burnout and Dropout Risk Dataset](https://www.kaggle.com/datasets/harpartapsingh13/student-burnout-and-dropout-risk-dataset) on Kaggle. The analysis practices data cleaning, testing and modeling. Results describe the generated data and shouldn't be used for real decisions about students.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("student_burnout_dropout_dataset_2.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Student_ID              800 non-null    str    
 1   Age                     800 non-null    int64  
 2   Gender                  770 non-null    str    
 3   Year_of_Study           800 non-null    int64  
 4   Department              777 non-null    str    
 5   Residence_Type          783 non-null    str    
 6   Attendance_Percent      749 non-null    float64
 7   Study_Hours_Per_Day     765 non-null    float64
 8   Previous_GPA            765 non-null    float64
 9   Backlogs                785 non-null    float64
 10  Sleep_Hours             765 non-null    float64
 11  Screen_Time_Hours       764 non-null    float64
 12  Exercise_Freq_Per_Week  800 non-null    int64  
 13  Social_Activity_Score   760 non-null    float64
 14  Part_Time_Job           775 non-null    str    
 15  

In [3]:
missing = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_%": df.isna().mean() * 100,
}).sort_values('missing_%', ascending=False)
missing[missing["missing"] > 0]

,dtype,missing,missing_%
Attendance_Percent,float64,51,6.375
Stress_Level,float64,49,6.125
Family_Support_Score,float64,44,5.500
Financial_Stress_Score,float64,43,5.375
Motivation_Score,float64,41,5.125
Peer_Pressure_Score,float64,41,5.125
Social_Activity_Score,float64,40,5.000
Anxiety_Score,float64,39,4.875
Screen_Time_Hours,float64,36,4.500
Sleep_Hours,float64,35,4.375


In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,800.0,20.883750,2.602153,17.00,19.000,21.00,23.000,25.0
Year_of_Study,800.0,2.281250,1.105068,1.00,1.000,2.00,3.000,4.0
Attendance_Percent,749.0,76.884246,14.258149,30.00,66.200,77.40,88.600,100.0
Study_Hours_Per_Day,765.0,3.158431,1.558711,0.00,2.100,3.10,4.200,8.1
Previous_GPA,765.0,6.859242,1.401110,2.42,5.880,6.86,7.800,10.0
Backlogs,785.0,0.824204,0.939320,0.00,0.000,1.00,1.000,6.0
Sleep_Hours,765.0,6.199869,1.499472,2.00,5.200,6.30,7.200,10.0
Screen_Time_Hours,764.0,5.432461,2.200261,0.50,3.975,5.45,6.925,12.6
Exercise_Freq_Per_Week,800.0,2.011250,1.480325,0.00,1.000,2.00,3.000,7.0
Social_Activity_Score,760.0,4.994605,1.961375,0.00,3.600,5.00,6.400,10.0


There doesn't appear to be any impossible data. GPA is on a 0–10 scale (likely an Indian CGPA, given terms like Backlogs and PG/Rented), and no values fall outside it.

In [5]:
for col in df.select_dtypes(include="str").columns.drop("Student_ID"):
    print(df[col].value_counts(dropna=False), "\n")

Gender
Female    392
Male      346
Other      32
NaN        30
Name: count, dtype: int64 

Department
Engineering    209
Business       151
Science        143
Arts           136
Medicine        77
Law             61
NaN             23
Name: count, dtype: int64 

Residence_Type
Hostel         290
Day Scholar    286
PG/Rented      207
NaN             17
Name: count, dtype: int64 

Part_Time_Job
No     547
Yes    228
NaN     25
Name: count, dtype: int64 

Family_Income_Bracket
Middle          242
Lower-Middle    227
Upper-Middle    131
Low              95
High             92
NaN              13
Name: count, dtype: int64 

Counseling_Access
No     535
Yes    265
Name: count, dtype: int64 

Burnout_Level
Low       320
Medium    280
High      200
Name: count, dtype: int64 

Dropout_Risk
Yes    437
No     363
Name: count, dtype: int64 



In [6]:
clean = df.copy()

In [7]:
# Fills in missing Text column answers with Unknown
text_cols = ['Gender', 'Department', 'Residence_Type', 'Part_Time_Job', 'Family_Income_Bracket']
for col in text_cols:
    clean[col] = clean[col].fillna('Unknown')

#Fills inthe numeric column answers with the column's median
num_cols = clean.select_dtypes("number").columns
for col in num_cols:
    clean[col] = clean[col].fillna(clean[col].median())

In [8]:
print(clean.isna().sum().sum())
print(clean['Backlogs'].unique())

0
[2. 0. 1. 3. 4. 6. 5.]


For missing text answers I replaced them with the value 'Unknown' and for numeric values I used each column's median. Burnout level and dropout risk were not used for filling, so the cleaning doesn't build in the relationships I am looking to test. 

In [9]:
def dropout_rate_by(data, col):
    """Dropout rate (%) and student count for each group in a column."""
    return (data.groupby(col)["Dropout_Risk"]
                .agg(dropout_rate=lambda s: (s == "Yes").mean() * 100, students="count")
                .round(1)
                .sort_values("dropout_rate", ascending=False))

In [10]:
from scipy import stats

def dropout_p_value(data, col, exclude=None):
    """p-value for whether dropout rates differ across the groups in a column."""
    d = data if exclude is None else data[~data[col].isin(exclude)]
    table = pd.crosstab(d[col], d["Dropout_Risk"])
    return stats.chi2_contingency(table)[1]

In [11]:
display(dropout_rate_by(clean, 'Counseling_Access'))
dropout_rate_by(clean, 'Part_Time_Job')

,dropout_rate,students
Counseling_Access,,
No,57.6,535
Yes,48.7,265


,dropout_rate,students
Part_Time_Job,,
Unknown,64.0,25
Yes,62.3,228
No,51.0,547


In [12]:
dropout_rate_by(clean, 'Burnout_Level')

,dropout_rate,students
Burnout_Level,,
High,92.0,200
Medium,63.2,280
Low,23.8,320


Burnout shows the strongest link of any factor: 92.0% of high-burnout students are at dropout risk, compared with 23.8% of low-burnout students, a 68.2 percentage point gap (p < 0.001). The rate rises sharply at each level, from 23.8% (low) to 63.2% (medium) to 92.0% (high).

Students with access to counseling have an 8.9 percentage point lower dropout risk rate than those without (48.7% vs. 57.6%, p = 0.021), a modest but statistically significant difference.

Students with a part-time job have an 11.3 percentage point higher dropout risk rate than those without (62.3% vs. 51.0%, p = 0.005).


In [13]:
clean["Family_Support_Group"] = pd.qcut(clean["Family_Support_Score"], 3, labels=["Low", "Medium", "High"])
dropout_rate_by(clean, "Family_Support_Group")

,dropout_rate,students
Family_Support_Group,,
Low,63.4,273
Medium,53.5,269
High,46.5,258


Students with low family support have a dropout risk rate of 63.4%, compared with 46.5% for those with high support, a 16.9 percentage point gap (p < 0.001). The rate falls steadily as support increases, so family support is clearly linked to lower dropout risk, though this data can’t show that it causes it.

In [14]:
checks = [("Counseling_Access", None), ("Part_Time_Job", ["Unknown"]),
          ("Family_Support_Group", None), ("Burnout_Level", None)]
for col, exclude in checks:
    p = dropout_p_value(clean, col, exclude)
    print(f"{col}: p < 0.001" if p < 0.001 else f"{col}: p = {p:.3f}")

Counseling_Access: p = 0.021
Part_Time_Job: p = 0.005
Family_Support_Group: p < 0.001
Burnout_Level: p < 0.001


### Q2: Which habits are linked to burnout?

In [15]:
habits = ["Sleep_Hours", "Exercise_Freq_Per_Week", "Social_Activity_Score",
          "Study_Hours_Per_Day", "Screen_Time_Hours"]
levels = ["Low", "Medium", "High"]

rows = []
for h in habits:
    groups = [clean.loc[clean["Burnout_Level"] == lvl, h] for lvl in levels]
    p = stats.kruskal(*groups).pvalue
    rows.append([h] + [g.median() for g in groups] + ["< 0.001" if p < 0.001 else round(p, 3)])

burnout_habits = pd.DataFrame(rows, columns=["habit", "Low burnout", "Medium burnout", "High burnout", "p-value"])
burnout_habits

,habit,Low burnout,Medium burnout,High burnout,p-value
0,Sleep_Hours,6.3,6.300,6.20,0.183
1,Exercise_Freq_Per_Week,2.0,2.000,2.00,0.006
2,Social_Activity_Score,5.1,5.000,4.90,0.127
3,Study_Hours_Per_Day,3.4,3.100,2.75,< 0.001
4,Screen_Time_Hours,5.1,5.475,5.85,< 0.001


In [16]:
# Share of students with 0 exercise sessions per week, by burnout level
never_exercise = (clean.groupby("Burnout_Level")["Exercise_Freq_Per_Week"]
                       .apply(lambda s: (s == 0).mean() * 100)
                       .reindex(["Low", "Medium", "High"])
                       .round(1))
never_exercise

Burnout_Level
Low       11.6
Medium    16.4
High      17.0
Name: Exercise_Freq_Per_Week, dtype: float64

In [17]:
# Average of each habit at each burnout level
habit_means = (clean.groupby("Burnout_Level")[habits]
                    .mean()
                    .reindex(["Low", "Medium", "High"])
                    .round(2)
                    .T)
habit_means

Burnout_Level,Low,Medium,High
Sleep_Hours,6.29,6.23,6.02
Exercise_Freq_Per_Week,2.19,1.96,1.80
Social_Activity_Score,5.15,4.95,4.81
Study_Hours_Per_Day,3.58,3.06,2.62
Screen_Time_Hours,4.92,5.60,6.03


#### Q2 findings

Study hours and screen time show the clearest links. High-burnout students study less and spend more time on screens:

- **Study hours:** the median falls from 3.4 hours per day (low burnout) to 2.75 hours (high burnout), about 39 minutes less per day (p < 0.001).
- **Screen time:** the median rises from 5.1 hours per day (low burnout) to 5.85 hours (high burnout), about 45 minutes more per day (p < 0.001).

Exercise has a smaller but real link (p = 0.006). The median is 2 sessions per week at every level, because exercise is counted in whole sessions. But the average falls from 2.19 sessions (low burnout) to 1.80 (high burnout), and more high-burnout students never exercise (17.0% vs. 11.6%).

Sleep and social activity show no clear link. Median sleep differs by only 0.1 hours across burnout levels (p = 0.183), and social activity by only 0.2 points (p = 0.127).

Overall: burnout is linked most clearly to how students spend their day (less studying, more screen time) rather than to sleep or social life. This data can’t show which comes first. Students may burn out because of these habits, or the habits may change because students are already burned out.

### Q5: Are students with more backlogs more likely to be at dropout risk?

In [18]:
dropout_rate_by(clean, "Backlogs").sort_index()

,dropout_rate,students
Backlogs,,
0.0,53.5,346
1.0,53.6,304
2.0,58.7,109
3.0,62.1,29
4.0,62.5,8
5.0,66.7,3
6.0,0.0,1


In [19]:
clean["Backlog_Group"] = pd.cut(clean["Backlogs"], bins=[-1, 0, 1, 2, 10],
                                labels=["0", "1", "2", "3+"])
display(dropout_rate_by(clean, "Backlog_Group").sort_index())

p = dropout_p_value(clean, "Backlog_Group")
print(f"p < 0.001" if p < 0.001 else f"p = {p:.3f}")

,dropout_rate,students
Backlog_Group,,
0,53.5,346
1,53.6,304
2,58.7,109
3+,61.0,41


p = 0.634


In [20]:
clean.groupby("Year_of_Study")["Backlogs"].agg(["mean", "median", "count"]).round(2)

,mean,median,count
Year_of_Study,,,
1,0.81,1.0,258
2,0.86,1.0,208
3,0.88,1.0,185
4,0.76,1.0,149


No clear link. Dropout risk rises slightly with backlogs, from 53.5% (0 backlogs) to 61.0% (3+ backlogs), but the difference is not statistically significant (p = 0.634). The 3+ group has only 41 students, so a gap this size could easily be chance.

Backlogs also don’t build up over time. The average stays around 0.8 backlogs per student in every year of study (0.76–0.88), with a median of 1 each year.

This was unexpected. In real universities, failed courses usually pile up and raise dropout risk. Finding no link here suggests either that backlogs matter less than expected for these students, or that the dataset doesn’t capture this relationship.

In [21]:
r = stats.spearmanr(clean["Backlogs"], clean["Previous_GPA"])
print(f"correlation = {r.statistic:.3f}, p = {r.pvalue:.3f}")

correlation = 0.006, p = 0.855


Backlogs and GPA are essentially unrelated (correlation 0.006), which is unrealistic and fits with the data being synthetic

In [22]:
clean["Attendance_Group"] = pd.qcut(clean["Attendance_Percent"], 3, labels=["Low", "Medium", "High"])
display(dropout_rate_by(clean, "Attendance_Group"))
print(dropout_p_value(clean, "Attendance_Group"))

,dropout_rate,students
Attendance_Group,,
Low,70.1,268
Medium,50.9,265
High,42.7,267


4.994910954840969e-10

Attendance is strongly linked to dropout risk. 70.1% of students with low attendance are at risk, compared with 42.7% with high attendance, a 27.4 percentage point gap (p < 0.001). The rate falls steadily as attendance rises, and attendance was also the top factor in the model once burnout was removed.

## The Model

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance

# Outcome: 1 = at risk, 0 = not at risk
y = (df["Dropout_Risk"] == "Yes").astype(int)
# Features: everything except the ID and the outcome
X = df.drop(columns=["Student_ID", "Dropout_Risk"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

In [24]:
def build_models(X):
    num_cols = X.select_dtypes("number").columns
    cat_cols = X.select_dtypes(exclude="number").columns

    def prep():
        return ColumnTransformer([
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
            ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="Unknown"),
                                  OneHotEncoder(handle_unknown="ignore")), cat_cols),
        ])

    return {
        "Baseline (always 'Yes')": DummyClassifier(strategy="most_frequent"),
        "Logistic regression": make_pipeline(prep(), LogisticRegression(max_iter=1000)),
        "Random forest": make_pipeline(prep(), RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5, random_state=42)),
    }

In [25]:
def compare_models(X_tr, X_te):
    models = build_models(X_tr)
    for name, model in models.items():
        model.fit(X_tr, y_train)
        print(f"{name}: accuracy = {model.score(X_te, y_test):.3f}")
    return models

models = compare_models(X_train, X_test)

Baseline (always 'Yes'): accuracy = 0.545
Logistic regression: accuracy = 0.750


Random forest: accuracy = 0.755


In [26]:
def importance(model, X_te):
    result = permutation_importance(model, X_te, y_test, n_repeats=20, random_state=42)
    return (pd.Series(result.importances_mean, index=X_te.columns)
              .sort_values(ascending=False).round(3))

importance(models["Random forest"], X_test).head(10)

Burnout_Level           0.226
Motivation_Score        0.010
Anxiety_Score           0.003
Year_of_Study           0.002
Department              0.001
Part_Time_Job           0.001
Peer_Pressure_Score     0.001
Previous_GPA            0.001
Family_Support_Score    0.001
Backlogs                0.000
dtype: float64

In [27]:
X_train_nb = X_train.drop(columns="Burnout_Level")
X_test_nb = X_test.drop(columns="Burnout_Level")

models_nb = compare_models(X_train_nb, X_test_nb)
importance(models_nb["Random forest"], X_test_nb).head(10)

Baseline (always 'Yes'): accuracy = 0.545
Logistic regression: accuracy = 0.735


Random forest: accuracy = 0.705


Attendance_Percent        0.079
Study_Hours_Per_Day       0.045
Motivation_Score          0.036
Screen_Time_Hours         0.031
Financial_Stress_Score    0.028
Stress_Level              0.027
Anxiety_Score             0.026
Previous_GPA              0.025
Counseling_Access         0.022
Family_Support_Score      0.022
dtype: float64

#### Model findings

Both models clearly beat guessing. Always guessing “at risk” is right 54.5% of the time. With all factors, logistic regression reaches 75.0% accuracy and the random forest 75.5%, about 21 percentage points better than guessing.

Burnout dominates. Shuffling burnout lowers the random forest’s accuracy by 22.6 percentage points. No other factor lowers it by more than 1 point. In this synthetic dataset, dropout risk appears to have been built largely from burnout.

Without burnout, other factors take over. Accuracy drops to 73.5% (logistic regression) and 70.5% (random forest), still well above guessing. The top factors become attendance (7.9 points), study hours (4.5), motivation (3.6) and screen time (3.1). Burnout overlaps with these: Q2 showed high-burnout students study less and spend more time on screens. So when burnout is in the model, these factors add little new information.

Connecting back: family support and part-time jobs were clearly linked to dropout on their own (Q1, Q3), but add little once burnout is known. Backlogs matter in neither analysis, which matches Q5.

## Conclusion: What factors are linked to dropout risk?

- Burnout is the strongest factor by far. 92.0% of high-burnout students are at dropout risk, compared with 23.8% of low-burnout students, a 68.2 percentage point gap (p < 0.001). In the model, shuffling burnout lowers accuracy by 22.6 percentage points, more than 20 times any other factor.
- Attendance is the second strongest. 70.1% of low-attendance students are at risk vs. 42.7% of high-attendance students, a 27.4-point gap (p < 0.001). It’s also the top factor in the model once burnout is removed.
- Smaller but real links: low family support (+16.9 points vs. high support, p < 0.001), having a part-time job (+11.3 points, p = 0.005), and lacking counseling access (+8.9 points, p = 0.021).
- Study hours and screen time are the habits most linked to burnout. High-burnout students study about 39 minutes less per day and spend about 45 minutes more on screens (both p < 0.001).
- No clear link: backlogs (p = 0.634), sleep (p = 0.183) and social activity (p = 0.127).
- The model predicts dropout risk with 75.5% accuracy (random forest), about 21 percentage points better than always guessing “at risk” (54.5%).

## Limitations

- The data is synthetic ([Kaggle source](https://www.kaggle.com/datasets/harpartapsingh13/student-burnout-and-dropout-risk-dataset)), not real student records. The results reflect how the data was generated, which likely explains burnout’s dominance and the missing backlog–GPA link (correlation 0.006).
- The data shows links, not causes. For example, it can’t tell whether less studying and more screen time lead to burnout, or burnout leads to less studying and more screen time.
- The models were tested on one 25% split of 800 students. A different split could shift accuracy by a few points.